# 01 — Prepare Policy Documents

**Purpose:** Turn each structured insurance row into a retrievable natural-language
"policy profile" document, and derive the age/cost buckets that ground the classify and
analysis-report tools later in the pipeline.

### What This Notebook Does
1. Reads `raw.policies_raw` and derives `age_bucket` and `charges_tier` (Low/Medium/High,
   from the data's own 33rd/66th percentiles).
2. Text-serializes every row into a one-paragraph `policy_text` description — this is what
   makes a purely tabular dataset usable as a RAG corpus.
3. Writes `docs.policy_documents` with a primary key and Change Data Feed enabled, which
   Vector Search's Delta Sync Index requires downstream.

### Source & Target
| Direction | Table |
|-----------|-------|
| Source | `insurance_rag_agent.raw.policies_raw` |
| Target | `insurance_rag_agent.docs.policy_documents` |

> **Prerequisites:** Run `schema_mgt/00_setup_and_data_ingest` first.

---

In [0]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG      = 'insurance_rag_agent'
RAW_TABLE    = f'{CATALOG}.raw.policies_raw'
DOCS_TABLE   = f'{CATALOG}.docs.policy_documents'

# Idempotency — drop the target before re-running
spark.sql(f'DROP TABLE IF EXISTS {DOCS_TABLE}')

print(f'Source: {RAW_TABLE}')
print(f'Target: {DOCS_TABLE}')

In [0]:
from pyspark.sql import functions as F

---

In [0]:
df = spark.read.table(RAW_TABLE)

display(df.limit(5))

---

In [0]:
LOW_THRESHOLD, HIGH_THRESHOLD = df.approxQuantile('charges', [0.33, 0.66], 0.01)

bucketed_df = (
    df
    .withColumn('policy_id',
        F.monotonically_increasing_id())
    .withColumn('age_bucket',
        F.when(F.col('age') < 30, F.lit('18-29'))
         .when(F.col('age') < 45, F.lit('30-44'))
         .when(F.col('age') < 60, F.lit('45-59'))
         .otherwise(F.lit('60+')))
    .withColumn('charges_tier',
        F.when(F.col('charges') < LOW_THRESHOLD,  F.lit('Low'))
         .when(F.col('charges') < HIGH_THRESHOLD, F.lit('Medium'))
         .otherwise(F.lit('High')))
    .withColumn('smoker_desc',
        F.when(F.col('smoker') == 'yes', F.lit('a smoker'))
         .otherwise(F.lit('a non-smoker')))
)

out_df = (
    bucketed_df
    .withColumn('policy_text', F.format_string(
        'Policyholder %d: a %d-year-old %s from the %s region, %s, with a BMI of %.1f '
        'and %d dependent(s). Annual insurance charge: $%.2f.',
        F.col('policy_id'), F.col('age'), F.col('sex'), F.col('region'),
        F.col('smoker_desc'), F.col('bmi'), F.col('children'), F.col('charges')
    ))
    .withColumn('ingestion_timestamp', F.current_timestamp())
    .withColumn('source_file',         F.lit(RAW_TABLE))
    .select(
        'policy_id', 'policy_text', 'age', 'sex', 'bmi', 'children', 'smoker', 'region',
        'charges', 'age_bucket', 'charges_tier', 'ingestion_timestamp', 'source_file'
    )
)

display(out_df.select('policy_id', 'policy_text', 'age_bucket', 'charges_tier').limit(5))

---

In [0]:
(out_df.write.format('delta').mode('overwrite').saveAsTable(DOCS_TABLE))

# Vector Search's Delta Sync Index needs a primary key + Change Data Feed
spark.sql(f'ALTER TABLE {DOCS_TABLE} ALTER COLUMN policy_id SET NOT NULL')
spark.sql(f'ALTER TABLE {DOCS_TABLE} ADD CONSTRAINT policy_documents_pk PRIMARY KEY (policy_id)')
spark.sql(f'ALTER TABLE {DOCS_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)')

print(f'Rows written: {out_df.count()}')

---

In [0]:
%sql
SELECT policy_id, policy_text, age_bucket, charges_tier
FROM insurance_rag_agent.docs.policy_documents
ORDER BY policy_id
LIMIT 5

In [0]:
print(f'Total rows in {DOCS_TABLE}: {spark.read.table(DOCS_TABLE).count()}')